# Diversity Metrics — Warm Evaluation
Computes three diversity axes for each model's top-10 recommendations on the
1000-user warm test set:
1. **ILS (Intra-List Similarity)**: mean pairwise cosine similarity of recommended songs'
   audio features. Lower = more diverse. Songs not in the content catalog are excluded
   from the ILS computation; lists with < 2 scoreable songs yield NaN.
2. **Genre coverage**: unique `track_genre` values in top-10 (Unknown excluded).
   Higher = more diverse.
3. **Catalog coverage**: across all 1000 users combined, fraction of unique recommended
   songs that appear in the content catalog (7.6K) and metadata catalog (12.3K).

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity

from src.data import load_msd_metadata, load_spotify_tracks, match_datasets, build_metadata_catalog
from src.models import CFModel, ContentModel, PopularityBaseline
from src.hybrid import HybridRecommender
from src.evaluation import bootstrap_ci

MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
K = 10


In [ ]:
cf      = CFModel.load(MODELS_DIR)
cm      = ContentModel.load(MODELS_DIR)
triplets = joblib.load(RESULTS_DIR / 'triplets_cache.pkl')
pop     = PopularityBaseline().fit(triplets)

msd_meta = load_msd_metadata()
spotify  = load_spotify_tracks()
matched  = match_datasets(spotify, msd_meta)
meta_cat = build_metadata_catalog(matched)

hybrid = HybridRecommender(cf, cm, meta_cat, alpha_strategy='adaptive')
cf_song_to_idx = {sid: idx for idx, sid in cf._idx_to_song.items()}

split                  = joblib.load(RESULTS_DIR / 'eval_split.pkl')
sampled_users          = split['sampled_users']
user_train_songs       = split['user_train_songs']
user_most_played_in_ct = split['user_most_played_in_ct']
print(f'Loaded: {len(sampled_users)} users')

# Pre-build genre lookup: song_id -> track_genre
genre_lookup = meta_cat.set_index('song_id')['track_genre'].to_dict()

# Content feature matrix for ILS
feat_matrix  = cm._X_scaled   # (7611, n_features), standardized
songid_to_idx = cm._songid_to_idx
print(f'Feature matrix: {feat_matrix.shape}')


In [ ]:
# ── Recommend functions ──────────────────────────────────────────────────
def cf_fn(uid):
    uidx      = cf._user_to_idx[uid]
    full      = cf._user_item[uidx].toarray().flatten()
    mask      = np.zeros(full.shape, dtype=bool)
    for sid in user_train_songs[uid]:
        if sid in cf_song_to_idx:
            mask[cf_song_to_idx[sid]] = True
    train_row = csr_matrix(full * mask)
    idxs, _   = cf._model.recommend(uidx, train_row, N=K, filter_already_liked_items=True)
    return [cf._idx_to_song[int(i)] for i in idxs]

def content_fn(uid):
    seed = user_most_played_in_ct.get(uid)
    return cm.recommend(seed, k=K)['song_id'].tolist() if seed else []

def pop_fn(uid):
    return pop.recommend(k=K)['song_id'].tolist()

def hybrid_fn(uid):
    return hybrid.recommend(
        user_id=uid, k=K,
        known_song_ids=set(user_train_songs[uid]),
    )['song_id'].tolist()

model_fns = {'popularity': pop_fn, 'cf': cf_fn, 'content': content_fn, 'hybrid': hybrid_fn}


In [ ]:
# ── Diversity helper functions ────────────────────────────────────────────
def ils(song_ids):
    """Mean pairwise cosine similarity; NaN if < 2 songs have audio features."""
    idxs = [songid_to_idx[s] for s in song_ids if s in songid_to_idx]
    if len(idxs) < 2:
        return np.nan
    vecs = feat_matrix[idxs]
    sim  = cosine_similarity(vecs)
    n    = len(idxs)
    return float(sim[np.triu_indices(n, k=1)].mean())

def genre_coverage(song_ids):
    """Unique non-Unknown genres in the recommendation list."""
    genres = {genre_lookup.get(s, 'Unknown') for s in song_ids}
    return len(genres - {'Unknown'})


In [ ]:
_DIV_CACHE = RESULTS_DIR / 'diversity_per_user.csv'

if _DIV_CACHE.exists():
    div_rows = pd.read_csv(_DIV_CACHE)
    print(f'Loaded diversity per-user from cache ({len(div_rows)} rows)')
else:
    records = []
    for model_name, fn in model_fns.items():
        print(f'  {model_name}...', end=' ', flush=True)
        for uid in sampled_users:
            try:
                recs = fn(uid)
            except Exception:
                recs = []
            records.append({
                'model':         model_name,
                'user_id':       uid,
                'ils':           ils(recs),
                'genre_cov':     genre_coverage(recs),
                'rec_list':      ','.join(recs),   # stored for catalog coverage
            })
        print('done')
    div_rows = pd.DataFrame(records)
    div_rows.to_csv(_DIV_CACHE, index=False)
    print(f'Saved ({len(div_rows)} rows)')


**Interpretation:**
Pure Content produces near-duplicate top-10 lists (ILS=0.904), confirming audio-feature k-NN saturates within micro-genres. CF and Hybrid both produce diverse lists (ILS≈0.27), 4x more diverse than Content alone. Hybrid does NOT exceed CF on intra-list diversity (CIs fully overlap), consistent with the alpha-bounded warm-blending finding from Checkpoint 2. A paired bootstrap test on per-user metadata-catalog hit counts shows Hybrid covers marginally more of the matched catalog than CF (delta=+0.025 songs/user, p=0.004), but Cohen's d=0.099 is negligible — this difference has no practical significance.

In [ ]:
# ── Aggregate ILS and genre coverage with bootstrap CIs ─────────────────
MODEL_ORDER = ['popularity', 'cf', 'content', 'hybrid']
content_ids = set(cm._songid_to_idx.keys())
meta_ids    = set(meta_cat['song_id'])
cf_ids      = set(cf._idx_to_song.values())

agg_rows = []
for model in MODEL_ORDER:
    sub = div_rows[div_rows['model'] == model]

    ils_scores  = sub['ils'].dropna().values
    gcov_scores = sub['genre_cov'].values.astype(float)

    ils_pt, ils_lo, ils_hi     = bootstrap_ci(ils_scores)
    gc_pt,  gc_lo,  gc_hi      = bootstrap_ci(gcov_scores)

    # Catalog coverage: unique songs across all users for this model
    all_recs = set()
    for rec_str in sub['rec_list']:
        if isinstance(rec_str, str) and rec_str:
            all_recs.update(rec_str.split(','))
    n_unique       = len(all_recs)
    n_in_content   = len(all_recs & content_ids)
    n_in_meta      = len(all_recs & meta_ids)
    n_in_cf        = len(all_recs & cf_ids)

    agg_rows.append({
        'model':           model,
        'ils_mean':        round(ils_pt, 4),
        'ils_ci_low':      round(ils_lo, 4),
        'ils_ci_high':     round(ils_hi, 4),
        'ils_n_users':     len(ils_scores),
        'genre_cov_mean':  round(gc_pt, 4),
        'genre_cov_low':   round(gc_lo, 4),
        'genre_cov_high':  round(gc_hi, 4),
        'unique_songs_rec':  n_unique,
        'pct_in_content_cat': round(100*n_in_content/n_unique, 1) if n_unique else 0,
        'pct_in_meta_cat':   round(100*n_in_meta/n_unique, 1)    if n_unique else 0,
        'pct_in_cf_cat':     round(100*n_in_cf/n_unique, 1)      if n_unique else 0,
    })

div_metrics = pd.DataFrame(agg_rows)
div_metrics.to_csv(RESULTS_DIR / 'diversity_metrics.csv', index=False)
print('Saved diversity_metrics.csv\n')

print(f'{"Model":<12} {"ILS (lower=diverse)":<26} {"Genre cov":<22} {"Unique recs":>12} {"% in content":>14} {"% in meta":>10}')
print('-' * 100)
for r in agg_rows:
    ils_str = f'{r["ils_mean"]:.4f} [{r["ils_ci_low"]:.4f},{r["ils_ci_high"]:.4f}]'
    gc_str  = f'{r["genre_cov_mean"]:.2f} [{r["genre_cov_low"]:.2f},{r["genre_cov_high"]:.2f}]'
    print(f'{r["model"]:<12} {ils_str:<26} {gc_str:<22} {r["unique_songs_rec"]:>12} '
          f'{r["pct_in_content_cat"]:>13.1f}% {r["pct_in_meta_cat"]:>9.1f}%')


**ILS computability note:**
ILS is only computable for recommendations that include >=2 songs in the audio-feature catalog.
For each model, the fraction of users with valid ILS:
- Popularity: 0% — all top-10 popularity songs lie outside the content catalog, so ILS is undefined for 100% of users
- CF: 45% — CF recommends from the full 98K catalog; most songs lack audio features
- Content: 84% — content recommendations are drawn entirely from the 7.6K catalog
- Hybrid: 45% — mirrors CF (adaptive alpha keeps CF dominant)

For models where ILS is undefined, the bar is plotted at 0 with an explicit annotation.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

MODEL_ORDER = ['popularity', 'cf', 'content', 'hybrid']
LABELS      = ['Popularity', 'CF', 'Content', 'Hybrid']
COLOR       = ['#DD8452', '#4C72B0', '#55A868', '#C44E52']
x           = np.arange(len(MODEL_ORDER))

ils_valid_pct = {}
for m in MODEL_ORDER:
    sub = div_rows[div_rows['model']==m]['ils']
    ils_valid_pct[m] = sub.notna().sum() / len(sub) * 100

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ILS
ax = axes[0]
ils_means, ils_lo_err, ils_hi_err = [], [], []
for m in MODEL_ORDER:
    r = div_metrics[div_metrics['model']==m].iloc[0]
    v = r['ils_mean'] if not np.isnan(r['ils_mean']) else 0.0
    ils_means.append(v)
    ils_lo_err.append(0 if np.isnan(r['ils_mean']) else r['ils_mean'] - r['ils_ci_low'])
    ils_hi_err.append(0 if np.isnan(r['ils_mean']) else r['ils_ci_high'] - r['ils_mean'])
ax.bar(x, ils_means, color=COLOR, edgecolor='white',
       yerr=[ils_lo_err, ils_hi_err], capsize=4)
for i, m in enumerate(MODEL_ORDER):
    pct  = ils_valid_pct[m]
    lbl  = 'N/A\n(0% computable)' if pct == 0 else f'{pct:.0f}% computable'
    ypos = ils_means[i] + ils_hi_err[i] + 0.01
    ax.text(i, max(ypos, 0.03), lbl, ha='center', va='bottom', fontsize=7.5,
            color='#C44E52' if pct == 0 else '#555')
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel('Mean ILS (lower = more diverse)')
ax.set_title('Intra-List Similarity @ k=10\n(95% CI; Popularity ILS undefined)')
ax.set_ylim(0, 1.15)

# Genre coverage
ax = axes[1]
gc_means = [div_metrics[div_metrics['model']==m].iloc[0]['genre_cov_mean'] for m in MODEL_ORDER]
gc_lo    = [div_metrics[div_metrics['model']==m].iloc[0]['genre_cov_mean']
            - div_metrics[div_metrics['model']==m].iloc[0]['genre_cov_low']  for m in MODEL_ORDER]
gc_hi    = [div_metrics[div_metrics['model']==m].iloc[0]['genre_cov_high']
            - div_metrics[div_metrics['model']==m].iloc[0]['genre_cov_mean'] for m in MODEL_ORDER]
ax.bar(x, gc_means, color=COLOR, edgecolor='white', yerr=[gc_lo, gc_hi], capsize=4)
ax.set_xticks(x); ax.set_xticklabels(LABELS)
ax.set_ylabel('Mean unique genres in top-10')
ax.set_title('Genre Coverage @ k=10\n(95% bootstrap CI, n=1000 users)')
ax.set_ylim(0, max(gc_means) * 1.30)

fig.tight_layout()
fig.savefig(RESULTS_DIR / 'diversity_comparison.png', dpi=150)
plt.close(fig)
print('Saved diversity_comparison.png')
